# One Bundle, Three Outcomes: 0.24, 0.17, No Score

The public notebook [Gemma 4 Baseline | LB 0.24](https://www.kaggle.com/code/honghanhhh/gemma-4-baseline-lb-0-24) scored 0.24 for its author. We submitted its bundle **byte for byte** and got no score at all:

> *Your submission notebook exceeded the allowed runtime.*

Score noise between identical submissions is already well documented (see [Same Bytes, Two Scores](https://www.kaggle.com/code/victorzhong0110/same-bytes-two-scores) and [Most-Run Public Config Averages 10/58](https://www.kaggle.com/code/dariushafshar/gemma-4-most-run-public-config-averages-10-58)). This notebook is about the other failure: **the same bytes can also run out of time.**

**TL;DR**
- The 12-hour limit covers all ~120 hidden tasks, sandbox setup included, patch validation excluded. A bundle can only cap each task (`max_time_minutes`, `max_tool_calls`); there is no global deadline.
- We timed 101 local runs of this exact bundle. Scaled to a scorer that runs 1.6-1.7x slower than our setup, the 8-minute cap lands at about **12 h for 120 tasks**: right at the limit, so some draws finish and some do not.
- A **7-minute cap** buys about 40 minutes of margin there and costs at most one solved task in 30. A 6-minute cap buys about 1.5 h but costs three to five.
- Our 7-minute submission: the same bundle with only the time cap lowered to 7 minutes **finished and scored 0.15**. That is one draw: it shows the run fits in time, not that 7 minutes scores less than 8 (identical 8-minute bytes have drawn 0.24, 0.17 and nothing).


## What happened

All rows are our own submissions unless marked. "28 / 8" means `max_tool_calls: 28`, `max_time_minutes: 8`.

| Submission | Caps (calls / min) | Public LB |
|---|---|---|
| honghanhhh, *Gemma 4 Baseline \| LB 0.24* (author's draw) | 28 / 8 | 0.24 |
| The same bundle byte for byte (zip sha256 `205a677f…`), 2026-10-30 | 28 / 8 | **no score: exceeded the allowed runtime** |
| The same bundle with `max_time_minutes: 7` and the prompt line changed to match, 2026-11-02 | 28 / 7 | **0.15, finished in time** |
| The same bundle + one prompt line about scratch files, 2026-10-27 | 28 / 8 | 0.17 |
| Same caps, a prompt that saves tool calls, 2026-10-29 | 28 / 8 | 0.12 |
| 40 tool calls, 8 minutes, 2026-10-22 | 40 / 8 | **no score: exceeded the allowed runtime** |
| 40 tool calls, 6 minutes, thinking budget 1024 (two draws) | 40 / 6 | 0.12, 0.10 |

Two things follow. One file set gives 0.24, 0.17 or nothing, so a single draw says little. And the bundles that timed out were the ones whose caps allow the most time per task.


## How the timings were measured

- **Setup:** RTX PRO 6000 96 GB, vLLM 0.19.1, `gemma-4-31b-it-qat-w4a16-ct` with `max_model_len` 32768, `swegemma` 0.2.11 with the bundle's own `eval_config.yaml`, three tasks at a time. Running three tasks at once makes each model call about 1.4x slower than running one.
- **Tasks:** 101 tasks from the competition's training data (fastapi 59, rich 42).
- **Bundles:** `public_8min` is the LB-0.24 bundle unchanged. `callsaver_8min` is the same caps with a prompt that spends fewer tool calls.
- **`agent_s`:** seconds from the start of the task to its last tool call, read from the trace timestamps. The model's final turn after the last call is not included, so this slightly underestimates.
- **`last_edit_s`:** time of the last source edit (`edit_file`, `write_file`, or a shell command that writes files).

The data is embedded below, so the notebook needs no inputs.


In [ ]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV = """bundle,task,resolved,tool_calls,agent_s,last_edit_s,duration_s
public_8min,fastapi_15661,0,25,214.6,164.6,236.9
public_8min,fastapi_15588,0,28,324.1,247.2,332.9
public_8min,fastapi_15589,0,28,356.5,349.1,371.39
public_8min,fastapi_15030,0,14,186.3,175.7,195.79
public_8min,fastapi_15280,0,28,204.9,202.1,213.28
public_8min,fastapi_15023,0,24,170.1,155.0,178.05
public_8min,fastapi_14986,0,27,418.7,208.1,486.01
public_8min,fastapi_14978,0,28,125.5,119.7,136.03
public_8min,rich_4070,0,28,157.8,61.4,175.47
public_8min,rich_4077,0,28,388.6,388.6,393.93
public_8min,rich_4079,0,28,184.2,174.3,193.18
public_8min,rich_4076,0,28,205.6,,482.06
public_8min,rich_4075,0,28,191.1,190.0,464.6
public_8min,rich_4006,0,6,316.0,,591.7
public_8min,fastapi_14258,0,16,290.9,37.8,485.06
public_8min,fastapi_14873,0,15,478.2,471.4,482.7
public_8min,fastapi_13920,0,27,287.1,285.0,485.77
public_8min,fastapi_14794,1,17,149.4,81.9,156.54
public_8min,fastapi_14786,1,14,63.4,31.0,71.87
public_8min,fastapi_14492,1,14,42.9,31.8,49.29
public_8min,fastapi_14479,0,16,69.4,23.4,77.9
public_8min,fastapi_14463,1,17,72.7,60.7,87.8
public_8min,fastapi_12942,0,28,307.8,302.1,342.06
public_8min,fastapi_14099,0,28,461.8,436.7,469.68
public_8min,fastapi_14609,0,28,300.4,251.4,309.55
public_8min,fastapi_14372,1,27,133.8,90.5,140.92
public_8min,fastapi_14349,1,27,333.1,302.0,342.3
public_8min,fastapi_14512,0,28,133.1,13.1,156.79
public_8min,fastapi_14487,0,28,272.9,266.0,279.77
public_8min,fastapi_11355,0,28,216.6,195.4,227.83
public_8min,fastapi_14485,0,28,188.3,188.3,200.84
public_8min,fastapi_14482,0,28,223.7,220.7,319.74
public_8min,fastapi_14306,0,28,147.0,145.3,157.13
public_8min,fastapi_14458,1,12,189.2,148.5,195.53
public_8min,fastapi_14459,1,28,309.8,207.8,320.19
public_8min,fastapi_14448,0,14,337.8,281.0,349.92
public_8min,fastapi_14455,0,28,270.6,186.8,303.91
public_8min,fastapi_9753,0,28,233.9,192.7,242.46
public_8min,fastapi_14430,0,23,472.5,460.2,485.67
public_8min,fastapi_5077,1,28,221.2,133.1,251.7
public_8min,fastapi_9555,1,18,143.8,92.2,151.89
public_8min,fastapi_14361,1,12,115.8,93.0,120.87
public_8min,fastapi_13207,1,28,348.9,343.0,362.96
public_8min,fastapi_14360,1,26,316.5,236.0,326.06
public_8min,fastapi_14356,1,21,136.9,104.6,150.79
public_8min,fastapi_5624,1,24,237.5,179.3,245.08
public_8min,fastapi_14419,0,28,295.2,294.0,310.4
public_8min,fastapi_14301,1,24,125.4,89.5,134.86
public_8min,fastapi_14262,0,28,152.6,,170.18
public_8min,fastapi_14266,0,28,178.4,148.2,186.96
public_8min,fastapi_14246,0,28,210.3,192.0,221.3
public_8min,fastapi_9425,1,28,133.6,123.7,141.54
public_8min,fastapi_13713,0,28,208.9,199.2,214.55
public_8min,rich_3718,1,20,166.4,135.0,188.97
public_8min,rich_3934,0,27,478.3,478.3,486.51
public_8min,rich_3953,0,25,315.0,214.4,323.29
public_8min,rich_3944,0,16,233.3,199.6,238.48
public_8min,rich_3942,0,28,168.9,166.5,178.4
public_8min,rich_3938,0,28,253.9,164.6,263.66
public_8min,rich_3882,0,28,186.8,181.4,193.39
public_8min,rich_3894,1,14,102.2,67.0,127.19
public_8min,rich_3935,0,26,298.9,222.7,305.55
public_8min,rich_3905,1,14,62.0,53.7,67.32
public_8min,rich_3930,0,26,268.6,261.7,484.56
public_8min,fastapi_14791,0,15,166.0,120.8,173.54
public_8min,fastapi_15800,0,28,429.0,424.1,482.9
public_8min,fastapi_14371,0,24,170.3,127.6,385.69
public_8min,fastapi_14605,0,28,181.5,93.8,199.91
public_8min,fastapi_14583,0,28,247.6,240.2,257.16
public_8min,fastapi_14297,1,28,230.1,192.0,246.55
public_8min,fastapi_14303,1,28,255.4,247.4,278.91
public_8min,fastapi_13537,1,16,146.1,93.0,154.62
public_8min,fastapi_14953,0,28,191.1,191.1,216.34
public_8min,fastapi_13786,0,28,349.8,311.4,361.34
public_8min,fastapi_14616,0,28,222.7,161.2,234.14
public_8min,fastapi_11194,1,28,199.9,197.3,209.79
public_8min,rich_3180,0,26,475.0,447.0,484.92
public_8min,rich_3777,1,28,221.3,216.4,226.58
public_8min,rich_3468,0,28,279.8,232.7,287.36
public_8min,rich_3772,0,28,310.6,308.6,616.56
public_8min,rich_3675,0,20,161.6,96.1,166.84
public_8min,rich_3676,0,28,151.1,138.8,157.62
public_8min,rich_3518,1,24,98.8,71.3,103.02
public_8min,rich_3535,0,19,198.0,136.0,229.41
public_8min,rich_3521,1,28,240.1,231.0,247.26
public_8min,rich_3506,0,19,422.5,,482.02
public_8min,rich_3480,0,26,359.7,327.3,668.12
public_8min,rich_3472,0,28,168.5,167.8,264.38
public_8min,rich_3454,1,28,231.1,223.6,238.12
public_8min,rich_3471,0,28,265.4,263.4,272.24
public_8min,rich_3470,0,28,171.0,169.6,177.76
public_8min,rich_3052,0,11,403.5,385.2,409.37
public_8min,rich_3278,0,19,472.7,465.4,482.05
public_8min,rich_2943,1,22,335.7,53.9,343.13
public_8min,rich_2725,1,28,395.9,395.9,484.4
public_8min,rich_3105,0,28,323.7,323.7,329.7
public_8min,rich_3043,1,13,186.1,128.3,192.59
public_8min,rich_3067,1,15,59.8,50.0,63.77
public_8min,rich_3006,1,20,113.2,78.9,119.27
public_8min,rich_3063,0,28,462.6,461.3,468.31
public_8min,rich_3061,0,28,179.7,134.8,184.58
callsaver_8min,fastapi_15661,0,23,233.4,217.6,250.52
callsaver_8min,fastapi_15588,0,25,211.2,208.7,223.66
callsaver_8min,fastapi_15589,1,17,267.4,265.3,276.22
callsaver_8min,fastapi_15030,0,21,174.9,162.5,203.69
callsaver_8min,fastapi_15280,0,28,265.4,253.8,272.68
callsaver_8min,fastapi_15023,0,28,197.1,191.6,204.43
callsaver_8min,fastapi_14986,0,28,147.5,136.4,156.39
callsaver_8min,fastapi_14978,0,28,218.6,216.9,226.19
callsaver_8min,rich_4070,0,28,133.8,131.6,328.82
callsaver_8min,rich_4077,1,28,203.0,187.4,209.09
callsaver_8min,rich_4079,0,28,178.2,166.8,185.57
callsaver_8min,rich_4076,0,28,108.4,86.9,117.29
callsaver_8min,rich_4075,0,23,327.6,,572.6
callsaver_8min,rich_4006,0,4,273.3,,532.06
callsaver_8min,fastapi_14258,0,28,458.4,458.4,722.23
callsaver_8min,fastapi_14873,1,28,173.4,171.9,183.47
callsaver_8min,fastapi_13920,0,22,174.2,171.3,183.38
callsaver_8min,fastapi_14794,1,14,173.1,89.8,181.01
callsaver_8min,fastapi_14786,1,14,79.2,26.2,87.53
callsaver_8min,fastapi_14492,1,4,16.1,10.5,21.7
callsaver_8min,fastapi_14479,0,20,113.1,71.2,124.05
callsaver_8min,fastapi_14463,1,16,97.8,80.6,113.24
callsaver_8min,fastapi_12942,0,28,237.9,215.8,262.89
callsaver_8min,fastapi_14099,0,28,204.8,201.9,211.59
callsaver_8min,fastapi_14609,0,28,191.0,187.6,203.55
callsaver_8min,fastapi_14372,1,16,107.4,88.4,113.5
callsaver_8min,fastapi_14349,1,16,342.9,298.4,353.66
callsaver_8min,fastapi_14512,1,28,282.2,278.2,304.31
callsaver_8min,fastapi_14487,0,28,130.2,126.5,135.67
callsaver_8min,fastapi_11355,0,28,158.7,95.2,167.82
callsaver_8min,fastapi_14485,1,27,137.1,134.5,147.76
callsaver_8min,fastapi_14482,0,28,277.6,252.4,306.66
callsaver_8min,fastapi_14306,0,28,182.7,,189.07
callsaver_8min,fastapi_14458,1,12,194.3,137.7,203.08
callsaver_8min,fastapi_14459,0,28,370.0,369.3,380.71
callsaver_8min,fastapi_14448,0,17,284.8,256.9,296.36
callsaver_8min,fastapi_14455,0,28,195.6,153.8,203.39
callsaver_8min,fastapi_9753,1,28,233.2,227.5,241.68
callsaver_8min,fastapi_14430,0,16,416.0,,482.72
callsaver_8min,fastapi_5077,1,21,260.2,249.1,272.94
callsaver_8min,fastapi_9555,1,23,176.1,168.2,212.37
callsaver_8min,fastapi_14361,1,15,213.5,211.7,218.65
callsaver_8min,fastapi_13207,0,28,228.0,225.6,253.08
callsaver_8min,fastapi_14360,1,21,205.1,160.0,216.87
callsaver_8min,fastapi_14356,1,28,206.5,176.6,218.8
callsaver_8min,fastapi_5624,1,17,196.7,123.5,202.92
callsaver_8min,fastapi_14419,0,28,414.1,390.2,424.99
callsaver_8min,fastapi_14301,1,26,135.1,114.7,144.3
callsaver_8min,fastapi_14262,0,28,229.2,180.1,242.8
callsaver_8min,fastapi_14266,0,28,167.0,142.7,174.77
callsaver_8min,fastapi_14246,0,28,146.9,56.6,154.65
callsaver_8min,fastapi_9425,0,28,142.2,139.4,150.45
callsaver_8min,fastapi_13713,0,28,159.6,124.2,168.7
callsaver_8min,rich_3718,1,14,82.2,67.0,87.57
callsaver_8min,rich_3934,0,19,67.9,45.4,76.8
callsaver_8min,rich_3953,0,22,289.4,244.6,297.56
callsaver_8min,rich_3944,0,28,307.6,285.0,313.22
callsaver_8min,rich_3942,0,28,209.7,169.3,214.68
callsaver_8min,rich_3938,0,28,189.9,5.0,285.86
callsaver_8min,rich_3882,0,28,176.8,172.5,182.11
callsaver_8min,rich_3894,1,16,117.8,52.3,131.5
callsaver_8min,rich_3935,0,28,262.2,259.3,270.9
callsaver_8min,rich_3905,1,25,72.5,42.5,82.52
callsaver_8min,rich_3930,0,22,220.1,220.1,484.85
callsaver_8min,fastapi_14791,0,21,434.8,188.9,443.1
callsaver_8min,fastapi_15800,0,24,371.0,353.9,379.67
callsaver_8min,fastapi_14371,0,28,461.4,417.5,476.08
callsaver_8min,fastapi_14605,1,23,165.4,102.0,176.68
callsaver_8min,fastapi_14583,0,28,411.1,389.3,421.11
callsaver_8min,fastapi_14297,1,19,142.1,71.1,160.51
callsaver_8min,fastapi_14303,1,21,117.5,37.2,137.91
callsaver_8min,fastapi_13537,1,11,169.8,103.5,178.16
callsaver_8min,fastapi_14953,0,28,126.1,90.5,135.58
callsaver_8min,fastapi_13786,0,27,407.8,387.1,420.52
callsaver_8min,fastapi_14616,0,28,198.0,192.6,210.5
callsaver_8min,fastapi_11194,1,28,190.2,186.2,205.12
callsaver_8min,rich_3180,0,26,465.3,465.3,482.23
callsaver_8min,rich_3777,0,28,127.3,,131.64
callsaver_8min,rich_3468,0,27,310.2,304.8,317.29
callsaver_8min,rich_3772,0,24,454.4,423.0,758.66
callsaver_8min,rich_3675,0,20,137.1,97.3,142.77
callsaver_8min,rich_3676,0,27,216.0,196.5,223.87
callsaver_8min,rich_3518,1,20,100.1,81.4,106.8
callsaver_8min,rich_3535,0,17,267.8,264.7,275.23
callsaver_8min,rich_3521,0,28,247.4,247.4,519.88
callsaver_8min,rich_3506,0,3,26.3,,510.37
callsaver_8min,rich_3480,1,15,334.3,317.0,485.73
callsaver_8min,rich_3472,0,28,263.8,262.9,270.53
callsaver_8min,rich_3454,1,14,122.1,63.9,131.46
callsaver_8min,rich_3471,0,24,164.0,112.8,169.09
callsaver_8min,rich_3470,0,28,255.1,255.1,350.33
callsaver_8min,rich_3052,0,28,153.5,144.8,159.77
callsaver_8min,rich_3278,1,27,309.6,298.1,316.15
callsaver_8min,rich_2943,1,24,99.9,73.5,107.52
callsaver_8min,rich_2725,0,17,238.1,210.1,257.8
callsaver_8min,rich_3105,0,28,277.8,,287.02
callsaver_8min,rich_3043,1,25,289.9,242.2,295.17
callsaver_8min,rich_3067,1,15,64.5,51.7,68.83
callsaver_8min,rich_3006,1,28,137.7,131.4,148.43
callsaver_8min,rich_3063,0,26,322.3,307.3,344.61
callsaver_8min,rich_3061,0,25,163.0,118.3,169.31
"""
df = pd.read_csv(io.StringIO(CSV))
pub = df[df.bundle == "public_8min"].reset_index(drop=True)
cs = df[df.bundle == "callsaver_8min"].reset_index(drop=True)

summary = df.groupby("bundle").agg(
    tasks=("task", "size"),
    resolved=("resolved", "sum"),
    hit_28_calls=("tool_calls", lambda s: int((s >= 28).sum())),
    agent_mean_s=("agent_s", "mean"),
    agent_p90_s=("agent_s", lambda s: s.quantile(0.9)),
).round(0)
summary

Locally the 28-call cap stops most long tasks before the 8-minute cap does. Only a handful of tasks get near 8 minutes on our hardware:

In [ ]:
# Colors: validated light-mode reference palette (ordinal blue ramp for the caps)
SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
BLUE = "#2a78d6"
CAP_COLOR = {8: "#104281", 7: "#2a78d6", 6: "#86b6ef"}

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
})

fig, ax = plt.subplots(figsize=(9, 3.8))
bins = np.arange(0, 541, 30)
ax.hist(pub.agent_s, bins=bins, color=BLUE, edgecolor=SURFACE, linewidth=2)
ax.grid(axis="x", visible=False)
for m in (6, 7, 8):
    ax.axvline(m * 60, color=INK2, linewidth=1, linestyle=(0, (3, 3)))
    ax.text(m * 60 + 4, ax.get_ylim()[1] * 0.92, f"{m} min", color=INK2, fontsize=10)
ax.set_xlabel("agent time per task on our setup (seconds)")
ax.set_ylabel("tasks")
ax.set_title("LB-0.24 bundle: agent time per task, 101 local tasks", loc="left", color=INK, fontsize=12)
plt.tight_layout()
plt.show()
print(f"median {pub.agent_s.median():.0f} s, mean {pub.agent_s.mean():.0f} s, "
      f"above 6 min: {(pub.agent_s > 360).sum()}, above 7 min: {(pub.agent_s > 420).sum()}")

## Projecting the hidden run

The scorer is slower than our setup, and its speed is not published. Call the slowdown **k**: one second of agent work here takes k seconds there. For a cap of C minutes, each task costs `min(k * agent_s, 60 * C)` plus sandbox setup (10 s assumed), and the total is scaled from 101 to 120 tasks. A resolved task is kept if `k * last_edit_s` is still inside the cap.

The same bytes have both finished (0.24) and run out of time on the scorer, so for this bundle the projected total must sit near 12 h. The cell below finds the k where that happens.

In [ ]:
N_HIDDEN, SETUP_S, LIMIT_H = 120, 10, 12.0

def project(sub, cap_min, k):
    cap = 60 * cap_min
    hours = (np.minimum(k * sub.agent_s, cap) + SETUP_S).sum() / len(sub) * N_HIDDEN / 3600
    edit = sub.last_edit_s.fillna(0)
    kept = int((sub.resolved.astype(bool) & (k * edit <= cap)).sum())
    return hours, kept

ks = np.round(np.arange(1.0, 2.31, 0.01), 2)

def k_at_limit(sub, cap_min):
    for k in ks:
        if project(sub, cap_min, k)[0] >= LIMIT_H:
            return k
    return None

for name, sub in [("public_8min", pub), ("callsaver_8min", cs)]:
    print(name, {f"{c} min": k_at_limit(sub, c) for c in (8, 7, 6)})

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.6))
for cap in (8, 7, 6):
    hrs = [project(pub, cap, k)[0] for k in ks]
    ax.plot(ks, hrs, color=CAP_COLOR[cap], linewidth=2, label=f"{cap}-minute cap")
    ax.text(ks[-1] + 0.02, hrs[-1], f"{cap} min", color=INK2, va="center", fontsize=10)
ax.axhline(LIMIT_H, color=INK2, linewidth=1, linestyle=(0, (3, 3)))
ax.text(1.01, LIMIT_H + 0.15, "12 h limit", color=INK2, fontsize=10)
# dots mark where each cap's total reaches 12 h
for cap, dx, dy, ha in ((8, -0.03, 0.25, "right"), (7, 0.03, -0.3, "left")):
    kc = k_at_limit(pub, cap)
    ax.plot([kc], [project(pub, cap, kc)[0]], "o", markersize=8, color=CAP_COLOR[cap],
            markeredgecolor=SURFACE, markeredgewidth=2, zorder=3)
    ax.text(kc + dx, LIMIT_H + dy, f"k = {kc:.2f}", color=INK, fontsize=10, ha=ha, va="center")
ax.set_xlim(1.0, 2.42)
ax.set_ylim(7.8, 14.4)
ax.set_xlabel("scorer slowdown k (scorer seconds per second on our setup)")
ax.set_ylabel("projected hours for 120 tasks")
ax.set_title("LB-0.24 bundle: projected runtime by time cap (dots: where each cap hits 12 h)", loc="left", color=INK, fontsize=12)
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
rows = []
for k in (1.45, 1.6, 1.7, 1.8):
    for cap in (8, 7, 6):
        h, kept = project(pub, cap, k)
        h2, kept2 = project(cs, cap, k)
        rows.append({"k": k, "cap (min)": cap, "hours / 120": round(h, 1),
                     "margin to 12 h (min)": round((LIMIT_H - h) * 60),
                     "resolved kept (of 33)": kept,
                     "call-saving prompt: hours": round(h2, 1)})
pd.DataFrame(rows).set_index(["k", "cap (min)"])

## What to take from it

- **The 8-minute public config runs at the edge of the limit.** Near k = 1.6-1.7 its projected total is about 12 h. Small changes in scorer speed, or a hidden task set slightly slower than ours, decide whether a draw scores.
- **7 minutes is cheap insurance.** At k = 1.6-1.7 it saves 40-45 minutes and loses at most one solved task in 30: only tasks whose last edit comes after the cap are lost. The 7-minute total reaches 12 h only at k = 1.83.
- **6 minutes is safer still but costs three to five solved tasks in 30** at the same k.
- **Saving tool calls also saves time.** The call-saving prompt projects about 0.9 h shorter at every cap, which fits it finishing on the scorer with 8 minutes.
- **Raising `max_tool_calls` above 28 adds time.** At 40 calls, tasks that used to stop at 28 calls run on to the time cap instead; our 40-call / 8-minute bundle timed out.

**Applying it:** set `max_time_minutes: 7` in `eval_config.yaml`. If your prompt states the budget ("28 tool calls and 8 minutes"), change that number too, so the prompt agrees with the budget the harness appends.

## Caveats
- The hidden tasks come from other repositories. Their setup and test times differ from fastapi and rich, so treat k as a sizing knob, not a measurement of the scorer.
- `agent_s` stops at the last tool call, so the projection is slightly low. Each k shown is therefore a little optimistic.
- Each configuration has one or two draws on the LB, and two timeouts is a small sample.
